# SEN2SR comparison

Compares our trained `DSen2Net20m` (from `../../notebooks/06_aoi_inference.ipynb`'s Western Region run) against [ESAOpenSR/SEN2SR](https://github.com/ESAOpenSR/SEN2SR)'s `Reference_RSWIR_x2` model — the SEN2SR variant that does the same 20m-bands-to-10m task ours does, not the more aggressive 4x-to-2.5m variant.

**Run this in the separate `sen2sr_compare` conda env** (see `README.md`), not the main project env — see the README for why.

**Real uncertainty flagged upfront**: I built this from SEN2SR's documented API without being able to run it myself (no network access). The exact input band order/count for `Reference_RSWIR_x2`'s "+ reference bands" isn't fully specified in what I could find — so rather than guess, the `inspect-metadata` cell below reads the downloaded model's own `mlm.json` (likely following the [STAC ML Model extension](https://github.com/stac-extensions/mlm) spec, given the filename) to discover the real expected input directly, before constructing anything. If that metadata doesn't clarify it, expect a shape-mismatch error on the first real run — normal for a first attempt at an unfamiliar API, not a sign anything is fundamentally wrong.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import mlstac
import numpy as np
import rasterio
import rasterio.windows
import torch

REPO_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from s2sr import config, stac

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Download the SEN2SR model

`Reference_RSWIR_x2` — the 20m-bands-to-10m variant, matching our own model's task. Downloaded once to `model/`, cached locally after that (see `.gitignore` — this directory is deliberately not tracked, it's a large downloaded model, not our own code).

In [ ]:
mlstac.download(
    file="https://huggingface.co/tacofoundation/sen2sr/resolve/main/SEN2SRLite/Reference_RSWIR_x2/mlm.json",
    output_dir="model/SEN2SRLite_Reference_RSWIR_x2",
)
sen2sr_model = mlstac.load("model/SEN2SRLite_Reference_RSWIR_x2").compiled_model(device=device)
print("Model loaded")

In [ ]:
mlm_path = Path("model/SEN2SRLite_Reference_RSWIR_x2/mlm.json")
mlm = json.loads(mlm_path.read_text())
print(json.dumps(mlm, indent=2)[:3000])  # print the metadata directly -- this is what actually
# determines the expected input band order/count/normalization below, not a guess.

## Load the same scene notebook 6 used

One representative tile from the Western Region, January 2026 run — reusing our own already-working `s2sr.stac`/boundary code rather than SEN2SR's suggested `cubo` package (see README).

In [ ]:
import geopandas as gpd

region_boundary_path = REPO_ROOT / "data" / "boundaries" / "ghana_western_region.geojson"
region_boundary = gpd.read_file(region_boundary_path)

items = stac.search_scenes(bbox=tuple(region_boundary.total_bounds), datetime_range="2026-01-01/2026-01-31")
item = items[0]  # match whichever scene notebook 6 actually picked, if this differs
print(f"Scene: {item.id}")

band_hrefs = {b: item.assets[b].href for b in [*config.BANDS_10M, *config.BANDS_20M]}
window = rasterio.windows.Window(0, 0, config.PATCH_SIZE_10M, config.PATCH_SIZE_10M)
native_20m_window = rasterio.windows.Window(
    0, 0, config.PATCH_SIZE_10M // config.DOWNSAMPLE_FACTOR_20M, config.PATCH_SIZE_10M // config.DOWNSAMPLE_FACTOR_20M
)

bands_10m = {b: rasterio.open(band_hrefs[b]).read(1, window=window).astype("float32") for b in config.BANDS_10M}
bands_20m = {b: rasterio.open(band_hrefs[b]).read(1, window=native_20m_window).astype("float32") for b in config.BANDS_20M}
print({b: arr.shape for b, arr in {**bands_10m, **bands_20m}.items()})

## Run SEN2SR

Band order/stacking here is a first guess (10m guide bands, then 20m bands, both /10000 normalized per SEN2SR's documented [0,1] input range) — check against what `inspect-metadata` above actually showed, and adjust if it doesn't match.

In [ ]:
guide_10m = np.stack([bands_10m[b] for b in config.BANDS_10M]) / 10_000.0
native_20m = np.stack([bands_20m[b] for b in config.BANDS_20M]) / 10_000.0

X = torch.from_numpy(np.concatenate([guide_10m, native_20m], axis=0)).float().to(device)
print(f"Input shape: {tuple(X.shape)}")

with torch.no_grad():
    sen2sr_output = sen2sr_model(X[None]).squeeze(0).cpu().numpy()
print(f"Output shape: {sen2sr_output.shape}")

## Visual comparison

Bicubic baseline vs. our DSen2 model vs. SEN2SR, same tile, same band (B11) — adjust the SEN2SR band index once `inspect-metadata` clarifies its actual output band order.

In [ ]:
from s2sr.dataset import upsample_bicubic

b11_20m = bands_20m["B11"]
bicubic_baseline = upsample_bicubic(b11_20m[None], out_size=config.PATCH_SIZE_10M)[0]

our_output_path = REPO_ROOT / "data" / "processed" / "superresolved" / "western_region_2026-01.tif"
with rasterio.open(our_output_path) as dst:
    our_band_idx = len(config.BANDS_10M) + config.BANDS_20M.index("B11") + 1
    our_output = dst.read(our_band_idx, window=window)

sen2sr_b11 = sen2sr_output[config.BANDS_20M.index("B11")]  # index guess -- see run-sen2sr-note

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, title in zip(
    axes, [bicubic_baseline, our_output, sen2sr_b11], ["bicubic baseline", "our DSen2 model", "SEN2SR"]
):
    ax.imshow(img, cmap="gray")
    ax.set_title(title, fontsize=10)
    ax.axis("off")
fig.tight_layout()

## Next steps

- If shapes/bands don't match on first run, the `inspect-metadata` cell's printed `mlm.json` is the source of truth to fix `run-sen2sr`/`compare-visual` against, not this notebook's initial guesses.
- This is one tile, one visual check — not a rigorous benchmark. If SEN2SR looks meaningfully better (or worse), decide from there whether it's worth a fairer comparison (e.g. same PSNR/SAM-style metrics notebook 4 uses, if SEN2SR's output can be validated against something).